Imports:

In [17]:
import folium
import sqlalchemy 
from sqlalchemy import create_engine, Column, Integer, String, Enum, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from folium import FeatureGroup 
from branca.colormap import linear
import pandas as pd
import os

# Generating data 


In [18]:


# Define the database URL (SQLite in this case)
DATABASE_URL = "sqlite:///pokemon.db"

# Create the database engine
engine = create_engine(DATABASE_URL)

# Define the base class for ORM models
Base = declarative_base()

# Define the Pokemon table as a Python class
class Pokemon(Base):
    __tablename__ = "pokemon"  # Name of the table
    id = Column(Integer, primary_key=True, autoincrement=True)
    name = Column(String(50), nullable=False)
    type = Column(Enum("fire", "ice", name="pokemon_type"), nullable=False) 
    location = Column(String(50), nullable=False)
    description = Column(Text, nullable=False)
    image = Column(String)

# Create the table in the database
Base.metadata.create_all(engine)

# Create a session for interacting with the database
Session = sessionmaker(bind=engine)
session = Session()

# List of Pokémon data entries (you can add more entries as needed)
pokemon_data = [
    {"name": "Charmander", "type": "fire", "location": '19.4219, -155.2872', "description": "A fire-type Pokémon.", "image": "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/4.png"},
    {"name": "Bulbasaur", "type": "fire", "location": '-82.8628, 135.0000', "description": "A grass-type Pokémon.", "image": "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/1.png"},
    {"name": "Squirtle", "type": "fire", "location": '-25.2744, 133.7751', "description": "A water-type Pokémon.", "image": "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/7.png"},
    {"name": "Articuno", "type": "ice", "location": '64.2008, -149.4937', "description": "An ice-type Legendary Pokémon.", "image": "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/144.png"},
    {"name": "Jynx", "type": "ice", "location": '64.9631, -19.0208', "description": "An ice and psychic-type Pokémon.", "image": "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/124.png"},
    
]

# Insert the Pokémon data into the database
for pokemon in pokemon_data:  # Use pokemon_data, not pokemon_entries
    new_pokemon = Pokemon(**pokemon)  # Unpack the dictionary directly
    session.add(new_pokemon)

# Commit the changes to the database
session.commit()

# Close the session
session.close()

/tmp/ipykernel_7830/233018641.py:8: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


# Creating a map

In [19]:

#We just change this to the actual sql table whenever that's done

DATABASE_URL = "sqlite:///pokemon.db"
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)
session = Session()


pokemon_entries = session.query(Pokemon).all()


map_center = [20.0, 0.0]
pokemon_map = folium.Map(location=map_center, zoom_start=2)


fire_group = FeatureGroup(name="Fire Pokémon")
ice_group = FeatureGroup(name="Ice Pokémon")


for pokemon in pokemon_entries:
   
    lat, lon = map(float, pokemon.location.split(","))
    popup = folium.Popup(pokemon.description, max_width=300)
    custom_icon = folium.CustomIcon(pokemon.image, icon_size=(80, 80)) 

    if pokemon.type == "fire":
        marker = folium.Marker(location=[lat, lon], popup=popup, icon=custom_icon)
        fire_group.add_child(marker)

    elif pokemon.type == "ice":
        marker = folium.Marker(location=[lat, lon], popup=popup, icon=custom_icon)
        ice_group.add_child(marker)

pokemon_map.add_child(fire_group)
pokemon_map.add_child(ice_group)
pokemon_map.add_child(folium.LayerControl())



# Initialize the base map centered between hottest and coldest places
m = folium.Map(location=[(df_hottest_places['latitude'].mean() + df_coldest_places['latitude'].mean()) / 2,
                         (df_hottest_places['longitude'].mean() + df_coldest_places['longitude'].mean()) / 2], zoom_start=2)

# Create color scales for hot and cold places
hot_colormap = linear.Reds_09.scale(df_hottest_places['temperature'].min(), df_hottest_places['temperature'].max())
cold_colormap = linear.Blues_09.scale(df_coldest_places['temperature'].min(), df_coldest_places['temperature'].max())

# Add markers for hottest places with gradient colors
for idx, row in df_hottest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='red', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=hot_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add markers for coldest places with gradient colors
for idx, row in df_coldest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='blue', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=cold_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add legends to the map
hot_colormap.caption = 'Temperature Scale (Hottest Places)'
cold_colormap.caption = 'Temperature Scale (Coldest Places)'
hot_colormap.add_to(m)
cold_colormap.add_to(m)


pokemon_map.save("pokemon_map.html")


session.close()



NameError: name 'df_hottest_places' is not defined

# Getting hottest and coldest places


In [24]:

# Load the JSON file 
file_path = "../Ecoregions_Coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)




filename_cold='/files/ds105a-2024-project-error_105/Temperature exploration/coldest_places.csv'
filename_hot= '/files/ds105a-2024-project-error_105/Temperature exploration/hottest_places.csv'

df_coldest_places=pd.read_csv(filename_cold)


# Drop the 'max_temperature' column
df_coldest_places = df_coldest_places.drop(columns=['max_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_coldest_places = pd.merge(df_coldest_places, coordinates_df, on="region", how="left")
df_coldest_places = df_coldest_places.rename(columns={'min_temperature': 'temperature'})



NameError: name 'coordinates' is not defined